In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import torch
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_undirected
import torch.serialization
from torch_geometric.data.data import Data, DataEdgeAttr

from GCN import GCN


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



device(type='cuda')

In [29]:
class GraphEncoderVAE(nn.Module):
    """
    Graph encoder for a VAE:
    - Takes a trace graph (x, edge_index, batch)
    - Outputs latent parameters (mu, logvar) and a sampled latent vector z
    """
    def __init__ (self, n_pods, n_ops, duration_mean, duration_std, hidden_ch=128, embed_dim=48, latent_dim=64, dropout=0.35):
        super().__init__()
        
        self.pod_embeddings = nn.Embedding(n_pods, embed_dim)
        self.op_embeddings = nn.Embedding(n_ops, embed_dim)
        self.duration_encoder = nn.Sequential(
            nn.Linear(1, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        
        in_channels = embed_dim * 3
        self.conv1 = GCNConv(in_channels, hidden_ch)
        self.bn1 = nn.BatchNorm1d(hidden_ch)
        self.conv2 = GCNConv(hidden_ch,hidden_ch)
        self.bn2 = nn.BatchNorm1d(hidden_ch)
        
         # Graph-level representation size (same as before)
        self.lin_feat = nn.Linear(hidden_ch, hidden_ch// 2)
        self.dropout = dropout

        # --- VAE latent heads ---
        self.latent_dim = latent_dim
        self.mu_lin = nn.Linear(hidden_ch // 2, latent_dim)
        self.logvar_lin = nn.Linear(hidden_ch // 2, latent_dim)

        # duration normalization as buffers
        duration_std = max(duration_std, 1e-6)
        self.register_buffer("duration_mean", torch.tensor(duration_mean))
        self.register_buffer("duration_std", torch.tensor(duration_std))
    
    def build_feats(self, x):
        """
        Same feature construction as your GCN:
        - x[:,0] = pod id (categorical)
        - x[:,1] = op id  (categorical)
        - x[:,2] = duration (continuous)
        """
        pod_ids = x[:, 0].long().clamp(min=0, max=self.pod_embeddings.num_embeddings - 1)
        op_ids = x[:, 1].long().clamp(min=0, max=self.op_embeddings.num_embeddings - 1)

        duration = torch.clamp(x[:,2], min=0)
        duration = torch.log1p(duration)
        duration = (duration - self.duration_mean) / (self.duration_std + 1e-9)
        duration = duration.unsqueeze(-1)

        pod_feat = self.pod_embeddings(pod_ids)
        op_feat = self.op_embeddings(op_ids)
        duration_feat = self.duration_encoder(duration)

        return torch.cat([pod_feat, op_feat, duration_feat], dim=1)
    
    def encode_graph(self, x, edge_idx, batch):
        """
        Core GNN encoder: returns a graph-level feature vector h_graph.
        """
        feats = self.build_feats(x)
        
        h = self.conv1(feats, edge_idx)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h,edge_idx)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # global graph representation
        h_graph = global_mean_pool(h, batch)  # [num_graphs, hidden_channels]
        h_graph = F.relu(self.lin_feat(h_graph))  # [num_graphs, hidden_channels//2]
        h_graph = F.dropout(h_graph, p=self.dropout, training=self.training)
        return h_graph
    
    def forward(self, x, edge_idx, batch):
        """
        Forward pass:
        returns (mu, logvar, z) for each graph in the batch.
        """
        h_graph = self.encode_graph(x, edge_idx, batch)
        
        mu = self.mu_lin(h_graph)
        logvar = self.logvar_lin(h_graph)

        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z = mu + eps * std
        else:
            z = mu
        
        return mu, logvar, z

    
    def encode(self, x, edge_idx, batch):
        """
        Convenience method: same as forward, but more explicit name.
        """ 
        
        return self.forward(x, edge_idx, batch)
        


In [30]:
class LoadDataset(InMemoryDataset):
    def __init__(self, datapath="./data.pt") -> None:
        torch.serialization.add_safe_globals([Data, DataEdgeAttr])
        super().__init__(".")
        data, slices = torch.load(datapath, weights_only=False)
        self.data, self.slices = data, slices
        x = data.x
        self.num_pods = int(x[:, 0].max().item()) + 1
        self.num_ops = int(x[:, 1].max().item()) + 1
        log_duration = torch.log1p(torch.clamp(x[:, 2], min=0))
        self.duration_mean = log_duration.mean().item()
        self.duration_std = log_duration.std().item() or 1.0

    def get(self, idx):
        data = super().get(idx)
        data.edge_index = to_undirected(data.edge_index)
        return data

In [31]:
def summarize_dataset(dataset):

    # num_graphs = data["y"].size(0)
    print("=== DATASET SUMMARY ===")
    print(f"Total graphs: {len(dataset)}")

    labels = []
    total_nodes = 0

    for data in dataset:
        if hasattr(data, "y") and data.y is not None:
            if data.y.numel() == 1:
                labels.append(int(data.y.item()))
            else:
                labels.extend(data.y.tolist())

        total_nodes += data.num_nodes

    if len(labels) == 0:
        print("No Labels found in dataset")
        return

    y = torch.tensor(labels)
    unique_labels, counts = torch.unique(y, return_counts=True)

    print("\nLabel distribution:")
    for label, count in zip(unique_labels.tolist(), counts.tolist()):
        pct = 100.0 * count / len(y)
        print(f"  Label {label}: {count} graphs ({pct:.1f}%)")

    print(f"\nNumber of unique labels: {len(unique_labels)}")
    print(f"Total nodes across all graphs: {total_nodes}")
    print(f"Average nodes per graph: {total_nodes / len(dataset):.2f}")


def split_dataset(dataset, train_ratio=0.8, seed=42):
    """
    Safely split a dataset into train and validation sets.
    Returns (train_dataset, val_dataset)
    """
    n_total = len(dataset)
    if n_total < 2:
        raise ValueError("Dataset must contain at least two graphs to split.")

    train_len = int(n_total * train_ratio)
    val_len = n_total - train_len  # ensures total matches exactly

    # Fix edge cases
    if train_len == 0:
        train_len = 1
        val_len = n_total - 1
    elif val_len == 0:
        val_len = 1
        train_len = n_total - 1

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
        dataset, [train_len, val_len], generator=generator
    )

    return train_dataset, val_dataset


def create_dataloaders(train_dataset, val_dataset, batch_size=32, num_workers=0):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    print(f"Dataloaders ready - batch size: {batch_size}")
    return train_loader, val_loader



In [33]:
def main():
    datapath = "./data.pt"
    dataset = LoadDataset(datapath)
    summarize_dataset(dataset)
    print(f"Unique Pods: {dataset.num_pods}, Unique Operations: {dataset.num_ops}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    encoder = GraphEncoderVAE(n_pods=dataset.num_pods, n_ops=dataset.num_ops, duration_mean=dataset.duration_mean, duration_std=dataset.duration_std, hidden_ch=128, embed_dim=48, latent_dim=64).to(device)
    
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    batch = next(iter(loader)).to(device)

    mu, logvar, z = encoder(batch.x, batch.edge_index, batch.batch)
    
    print(f"Mu Shape: {mu}]"
          f"\nLogVar Shape: {logvar.shape}"
          f"\nZ-shape: {z.shape}")


main()


=== DATASET SUMMARY ===
Total graphs: 2097

Label distribution:
  Label 0: 1089 graphs (51.9%)
  Label 1: 1008 graphs (48.1%)

Number of unique labels: 2
Total nodes across all graphs: 106020
Average nodes per graph: 50.56
Unique Pods: 28, Unique Operations: 200
Mu Shape: tensor([[-9.8416e-02,  3.2191e-01, -1.6351e-01,  1.3917e-02, -1.5136e-01,
          2.0147e-01, -2.8861e-01, -4.1498e-01, -1.2970e-02, -4.4137e-02,
         -4.0975e-02,  2.7891e-01,  2.2926e-02, -9.7040e-02,  8.8709e-02,
         -2.4640e-01, -3.1928e-01, -1.1664e-01, -1.4857e-01,  1.8699e-01,
          6.9421e-02,  2.1752e-01,  2.6056e-02, -8.6492e-02, -1.6406e-01,
          1.5764e-01,  1.3126e-01,  1.7601e-01, -7.7181e-02,  2.1821e-01,
          1.3420e-01, -5.7343e-02, -2.0668e-01, -1.2270e-01,  1.3130e-01,
          9.6715e-02,  1.3775e-01,  8.6291e-03,  9.2475e-02, -1.3745e-01,
          1.5088e-01, -1.6120e-01, -3.3096e-04, -2.4605e-01, -2.4463e-02,
          7.0132e-02, -2.2200e-01,  1.3532e-02, -1.8314e-01, 